## Executive Summary

This analysis examines the **ICE Brent Crude** forward curve structure over a 13-month period.

### Key Findings

1. **Moderate Supply Tightness**: Market in backwardation 88.9% of the time (256/288 days), indicating moderate inventory management

2. **Moderate Stress Event**: April 30, 2025 saw a 4-sigma event (Z-score = 3.96) with M1-M3 reaching 3.27 USD/bbl

3. **Normal Distribution**: Thin tails (kurtosis 1.37) - extreme events occur less frequently than normal distribution predicts

4. **Structure Drives Volatility**: Moderate correlation (0.52 for M1-M3) confirms physical tightness influences price instability

### Implication

Global crude oil market operates with more flexibility than diesel (LSGO). Brent shows less extreme stress, lower backwardation frequency, and thinner tails - consistent with a highly liquid global benchmark.

---

### The Brent Crude Market

**ICE Brent Crude** is the global oil benchmark, widely used for:
- Pricing ~70% of internationally traded crude oil
- Refinery feedstock
- Petrochemical production
- Energy security and strategic reserves

**Market characteristics:**
- Global benchmark (not regional like WTI)
- Highly liquid (largest oil futures market)
- Sensitive to geopolitical events (Middle East, Russia, OPEC)
- Storage capacity varies by region (tanks, floating storage)
- Influenced by OPEC+ production decisions

---

## 2. Data Preparation

We start by loading and cleaning the raw settlement data, then building rolling front-month contracts with proper expiry detection.

**Note on Units:**
- Brent Crude prices are in **USD/bbl** (barrels)
- LSGO (diesel) prices are in **USD/MT** (metric tons)
- This is normal - different commodities use different units
- Spreads and analysis methodology remain the same

---



**Note on Units:**
- Brent Crude prices are in **USD/bbl** (barrels)
- LSGO (diesel) prices are in **USD/MT** (metric tons)
- This is normal - different commodities use different units
- Spreads and analysis methodology remain the same

---



**Note on Units:**
- Brent Crude prices are in **USD/bbl** (barrels)
- LSGO (diesel) prices are in **USD/MT** (metric tons)
- This is normal - different commodities use different units
- Spreads and analysis methodology remain the same

---



### 2.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['grid.linewidth'] = 0.5

print('Libraries imported successfully')

### 2.2 Load Raw Data

The raw CSV contains:
- Daily settlement prices from January 2025 to February 2026
- Contract columns: **Feb-25, Mar-25, Apr-25, ..., Apr-26** (no Jan-25 - already expired)
- Clean settlements (no duplicates on expiry dates)
- Contracts expire at end of month, 2 months prior to delivery

---

In [ ]:
# Load raw data
file_path = 'datas/Brent_raw_2025_2026.csv'
df_raw = pd.read_csv(file_path)

# Display basic info
print(f'Raw data shape: {df_raw.shape}')
print(f'\nColumn names:')
print(df_raw.columns.tolist())
print(f'\nFirst few rows:')
df_raw.head(10)

### 2.3 Clean and Parse Data

In [ ]:
# Rename first column to Date
df_raw.columns = ['Date'] + df_raw.columns[1:].tolist()

# Parse dates
df_raw['Date'] = pd.to_datetime(df_raw['Date'], format='%m/%d/%Y')

# Sort by date
df_raw = df_raw.sort_values('Date').reset_index(drop=True)

# Convert price columns to numeric
contract_cols = df_raw.columns[1:].tolist()
for col in contract_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print('Data cleaned successfully')
print(f'Date range: {df_raw["Date"].min()} to {df_raw["Date"].max()}')
print(f'Number of trading days: {len(df_raw)}')
print(f'\nContract columns: {contract_cols}')
print(f'\nFirst 10 rows after cleaning:')
df_raw.head(10)

### 2.4 Inspect Data Quality

Check for missing values and understand the contract lifecycle.

In [ ]:
# Count non-null values per contract
print('Non-null values per contract:')
for col in contract_cols:
    count = df_raw[col].notna().sum()
    first_date = df_raw[df_raw[col].notna()]['Date'].min() if count > 0 else None
    last_date = df_raw[df_raw[col].notna()]['Date'].max() if count > 0 else None
    print(f'{col:10s}: {count:3d} days | First: {first_date} | Last: {last_date}')

### 2.5 Understand Roll Logic

**Brent Crude Expiry Rules:**

ICE Brent futures expire at the **end of the month, 2 months prior** to delivery:
- **Feb-25 contract**: Expires **January 31, 2025** (last day of month, 2 months before Feb)
- **Mar-25 contract**: Expires **February 28, 2025** (last day of month, 2 months before Mar)
- **Apr-25 contract**: Expires **March 31, 2025** (last day of month, 2 months before Apr)

**Key observations from our data:**
- **Jan-25 contract** is NOT in the data (expired November 30, 2024)
- **Feb-25** is M1 on January 2, 2025 (first trading day)
- **No duplicates** on expiry dates (clean settlements)
- Rolls occur at **beginning of each month** (after previous contract expires)

**Example:**
- January 31: Feb-25 last trading day (price = 74.92)
- February 3: Mar-25 is now M1 (Feb-25 disappeared)

---

In [ ]:
# Inspect potential roll dates
print('Example: Mar-25 contract (first available in Jan 2025)')
print('\nFirst days of data (Mar-25 should be M1):')
df_raw.head(5)[['Date', 'Mar-25', 'Apr-25', 'May-25']]

### 2.6 Build Rolling Contracts (M1 to M6)

We build a function that detects expired contracts and constructs M1-M6 for each trading day.

In [ ]:
def build_rolling_contracts(df, contract_cols, num_months=6):
    """
    Build rolling contracts for Brent Crude.
    
    Brent has clean settlements (no duplicates on expiry).
    We simply take the first num_months available contracts for each date.
    """
    rolling_data = []
    
    for idx, row in df.iterrows():
        date = row['Date']
        available_contracts = []
        
        # Get all non-null contracts for this date
        for col in contract_cols:
            if pd.notna(row[col]):
                available_contracts.append((col, row[col]))
        
        # Build M1-M6 from available contracts
        m_values = {'Date': date}
        for i in range(num_months):
            if i < len(available_contracts):
                m_values[f'M{i+1}'] = available_contracts[i][1]
                m_values[f'M{i+1}_contract'] = available_contracts[i][0]
            else:
                m_values[f'M{i+1}'] = np.nan
                m_values[f'M{i+1}_contract'] = None
        
        rolling_data.append(m_values)
    
    return pd.DataFrame(rolling_data)

df_rolling = build_rolling_contracts(df_raw, contract_cols)
print('Rolling contracts built (Brent clean settlement logic)')
df_rolling.head(10)

### 2.7 Detect Roll Dates

In [ ]:
df_rolling['M1_contract_prev'] = df_rolling['M1_contract'].shift(1)
df_rolling['is_roll'] = df_rolling['M1_contract'] != df_rolling['M1_contract_prev']
roll_dates = df_rolling[df_rolling['is_roll'] == True][['Date', 'M1_contract_prev', 'M1_contract', 'M1', 'M2']]
print('Roll dates:')
roll_dates.head(15)

### 2.8 Validate First Days

Check the first few days to confirm Feb-25 is M1 (Jan-25 already expired).

---

In [ ]:
# Validate first days and first roll
print('VALIDATION: First days (Mar-25 should be M1)')
print('='*70)
print('\nFirst 5 days:')
print(df_rolling.head(5)[['Date', 'M1_contract', 'M1', 'M2_contract', 'M2', 'M3_contract', 'M3']])

print('\n' + '='*70)
print('VALIDATION: First roll (Mar-25 to Apr-25 around Mar 3)')
print('='*70)
print('\nDates around first roll:')
print(df_rolling[(df_rolling['Date'] >= '2025-02-28') & (df_rolling['Date'] <= '2025-03-05')][['Date', 'M1_contract', 'M1', 'M2_contract', 'M2', 'is_roll']])

print('\n' + '='*70)
print('Expected:')
print('  - Jan 2 - Feb 28: M1 = Mar-25')
print('  - Mar 3+: M1 = Apr-25 (roll occurred)')
print('  - No Jan-25 or Feb-25 in data (already expired)')
print('='*70)

### 2.9 Calculate Spreads

In [ ]:
df_rolling['M1_M2'] = df_rolling['M1'] - df_rolling['M2']
df_rolling['M1_M3'] = df_rolling['M1'] - df_rolling['M3']
df_rolling['M1_M6'] = df_rolling['M1'] - df_rolling['M6']
print('Spreads calculated')
df_rolling[['M1_M2', 'M1_M3', 'M1_M6']].describe()

### 2.10 Calculate Returns and Volatility

In [ ]:
df_rolling['M1_return'] = np.log(df_rolling['M1'] / df_rolling['M1'].shift(1))
df_rolling['Vol_20D'] = df_rolling['M1_return'].rolling(window=20).std() * np.sqrt(252)
print('Volatility calculated')
df_rolling['Vol_20D'].describe()

### 2.11 Calculate Z-Scores

In [ ]:
df_rolling['M1_M3_mean_60'] = df_rolling['M1_M3'].rolling(window=60).mean()
df_rolling['M1_M3_std_60'] = df_rolling['M1_M3'].rolling(window=60).std()
df_rolling['Z_M1_M3'] = (df_rolling['M1_M3'] - df_rolling['M1_M3_mean_60']) / df_rolling['M1_M3_std_60']
df_rolling['M1_M6_mean_60'] = df_rolling['M1_M6'].rolling(window=60).mean()
df_rolling['M1_M6_std_60'] = df_rolling['M1_M6'].rolling(window=60).std()
df_rolling['Z_M1_M6'] = (df_rolling['M1_M6'] - df_rolling['M1_M6_mean_60']) / df_rolling['M1_M6_std_60']
print('Z-scores calculated')
df_rolling[['Z_M1_M3', 'Z_M1_M6']].describe()

### 2.12 Final Dataset

In [ ]:
print(f'Dataset shape: {df_rolling.shape}')
df_rolling[['Date', 'M1', 'M2', 'M3', 'M6', 'M1_M2', 'M1_M3', 'M1_M6', 'Vol_20D']].tail(10)

---

## 3. Data Validation

Before analyzing the data, we validate the rolling contract construction to ensure:
- Rolls are clean and coherent
- No artificial jumps from bad rolling logic
- Spreads and returns look realistic

### 3.1 Check for Artificial Jumps at Roll Dates

In [ ]:
# Calculate M1 price changes
df_rolling['M1_change'] = df_rolling['M1'].diff()
df_rolling['M1_pct_change'] = df_rolling['M1'].pct_change() * 100

# Identify large jumps on roll dates
roll_jumps = df_rolling[df_rolling['is_roll'] == True][['Date', 'M1_contract_prev', 'M1_contract', 'M1', 'M1_change', 'M1_pct_change']]
print('Price changes at roll dates:')
print(roll_jumps.head(15))
print(f'\nMean absolute roll jump: {roll_jumps["M1_pct_change"].abs().mean():.2f}%')
print(f'Max roll jump: {roll_jumps["M1_pct_change"].abs().max():.2f}%')

### 3.2 Inspect Suspicious Returns

In [ ]:
# Find extreme returns
extreme_returns = df_rolling[df_rolling['M1_pct_change'].abs() > 5][['Date', 'M1_contract', 'M1', 'M1_pct_change', 'is_roll']]
print('Days with extreme returns (>5%):')
print(extreme_returns)
print(f'\nTotal extreme return days: {len(extreme_returns)}')

### 3.3 Validate Spread Consistency

In [ ]:
# Check for unrealistic spread values
print('Spread validation:')
print(f'\nM1-M2 range: [{df_rolling["M1_M2"].min():.2f}, {df_rolling["M1_M2"].max():.2f}]')
print(f'M1-M3 range: [{df_rolling["M1_M3"].min():.2f}, {df_rolling["M1_M3"].max():.2f}]')
print(f'M1-M6 range: [{df_rolling["M1_M6"].min():.2f}, {df_rolling["M1_M6"].max():.2f}]')

# Check spread ordering (M1-M3 should generally be > M1-M2)
spread_order_check = (df_rolling['M1_M3'].abs() >= df_rolling['M1_M2'].abs()).sum()
print(f'\nDays where |M1-M3| >= |M1-M2|: {spread_order_check} / {len(df_rolling)} ({100*spread_order_check/len(df_rolling):.1f}%)')

### 3.4 Check Missing Values

In [ ]:
# Count missing values in key columns
print('Missing values:')
missing_cols = ['M1', 'M2', 'M3', 'M6', 'M1_M2', 'M1_M3', 'M1_M6', 'Vol_20D', 'Z_M1_M3', 'Z_M1_M6']
for col in missing_cols:
    missing_count = df_rolling[col].isna().sum()
    missing_pct = 100 * missing_count / len(df_rolling)
    print(f'{col:15s}: {missing_count:3d} ({missing_pct:5.1f}%)')

### 3.5 Validation Summary

In [ ]:
print('=== DATA VALIDATION SUMMARY ===')
print(f'\nTotal observations: {len(df_rolling)}')
print(f'Date range: {df_rolling["Date"].min()} to {df_rolling["Date"].max()}')
print(f'Number of rolls: {df_rolling["is_roll"].sum()}')
print(f'\nM1 price range: [{df_rolling["M1"].min():.2f}, {df_rolling["M1"].max():.2f}]')
print(f'Volatility range: [{df_rolling["Vol_20D"].min():.2f}, {df_rolling["Vol_20D"].max():.2f}]')
print(f'\nData quality: VALIDATED')
print('Ready for analysis and visualization.')

---

## 4. Market Analysis and Visualization

### Chart 1: Front-Month Price and Volatility

This chart shows M1 price dynamics alongside 20-day rolling volatility. Volatility spikes often coincide with structural tightening rather than purely macro noise.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 7))

# M1 price on left axis
ax1.plot(df_rolling['Date'], df_rolling['M1'], color='#1f77b4', linewidth=1.5, label='M1 Price')
ax1.set_xlabel('Date')
ax1.set_ylabel('M1 Price (USD/bbl)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, alpha=0.3)

# Volatility on right axis
ax2 = ax1.twinx()
ax2.plot(df_rolling['Date'], df_rolling['Vol_20D'], color='#d62728', linewidth=1.5, label='20D Volatility')
ax2.set_ylabel('Annualized Volatility', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')

# Format
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.title('LSGO Front-Month Price and Volatility', loc='left', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

### Analysis: Front-Month Price & Volatility Dynamics

**Key Observations:**

- **Price range**: 58.92 to 82.03 USD/bbl = **39.2% swing**
- **Volatility peak**: 52% annualized on Jul 08
- **Downward trend**: Jan high to Dec low (unlike LSGO V-shape)
- **Volatility-price relationship**: Vol spiked in July despite price decline

**Conclusion:** Brent showed downward price trend with moderate volatility. Different pattern from LSGO (which had V-shaped recovery).

---

**Analysis:**

The front-month price captures the market's prompt valuation. Volatility measures instability and tends to spike during periods of structural tightening, reflecting supply-demand imbalances rather than purely macro noise.

---

### Chart 2: Forward Curve Snapshots

Selected forward curve snapshots showing how the curve shape evolves through different market regimes.

In [ ]:
# Select key dates for curve snapshots
snapshot_dates = [
    '2025-01-15',  # Early period
    '2025-04-10',  # Post-crash
    '2025-06-19',  # Peak backwardation
    '2025-10-23',  # Stress regime
    '2026-01-30'   # Late period
]

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i, date_str in enumerate(snapshot_dates):
    date = pd.to_datetime(date_str)
    row = df_rolling[df_rolling['Date'] == date]
    if len(row) > 0:
        row = row.iloc[0]
        curve = [row['M1'], row['M2'], row['M3'], row['M4'], row['M5'], row['M6']]
        ax.plot([1,2,3,4,5,6], curve, marker='o', linewidth=2, color=colors[i], label=date_str)

ax.set_xlabel('Contract Month')
ax.set_ylabel('Price (USD/bbl)')
ax.set_title('Forward Curve Snapshots', loc='left', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xticks([1,2,3,4,5,6])
ax.set_xticklabels(['M1','M2','M3','M4','M5','M6'])
plt.tight_layout()
plt.show()

### Analysis: Forward Curve Evolution Across Regimes

**Key Observations:**

- **January 2025**: Mild backwardation (~1-2 USD/bbl) - normal regime
- **April-June 2025**: Spreads widened to 3-6 USD/bbl (moderate stress)
- **Peak**: June 19 with M1-M6 at 5.75 USD/bbl (still < 20 threshold)
- **Never reached Tight regime**: M1-M6 stayed below 20 USD/bbl throughout

**Conclusion:** Brent remained in Normal regime entire period. Much less stressed than LSGO.

---

**Analysis:**

- **Normal regime**: Mild backwardation or flatter curve
- **Stress regime**: Steep backwardation (M1 >> M6)
- **Normalization**: Curve flattening as supply improves

---

### Chart 3: Short-Term Structure (M1-M2 and M1-M3)

These spreads capture prompt physical tightness. Large positive spreads indicate scarce prompt barrels.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

ax.plot(df_rolling['Date'], df_rolling['M1_M2'], color='#1f77b4', linewidth=1.5, label='M1-M2', alpha=0.8)
ax.plot(df_rolling['Date'], df_rolling['M1_M3'], color='#ff7f0e', linewidth=1.5, label='M1-M3', alpha=0.8)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Date')
ax.set_ylabel('Spread (USD/bbl)')
ax.set_title('Short-Term Structure: M1-M2 and M1-M3', loc='left', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.show()

### Analysis: Short-Term Structure & Prompt Tightness

**Key Observations:**

- **M1-M3 average**: 1.14 USD/bbl, ranging from 0.00 to **3.27 USD/bbl**
- **Peak event**: M1-M3 at 3.27 USD/bbl on Jun 19 (2.9x average)
- **Backwardation dominant**: 88.9% of days (vs 99.7% for LSGO)
- **Moderate tightness**: Spreads much smaller than LSGO (3.27 vs 85.25 USD/MT)

**Conclusion:** Brent shows moderate prompt tightness. Global market more flexible than regional diesel.

---

**Analysis:**

- **M1-M2** captures immediate tightness
- **M1-M3** is a stronger signal of near-term structural imbalance
- Large positive spreads imply prompt barrels are scarce and physical supply cannot react quickly

---

### Chart 4: Long Structure (M1-M6)

This spread measures structural backwardation and storage economics.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

ax.plot(df_rolling['Date'], df_rolling['M1_M6'], color='#2ca02c', linewidth=1.5, label='M1-M6')
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Date')
ax.set_ylabel('Spread (USD/bbl)')
ax.set_title('Long Structure: M1-M6', loc='left', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.show()

### Analysis: Long Structure & Storage Economics

**Key Observations:**

- **M1-M6 average**: 2.25 USD/bbl (moderate backwardation)
- **Storage economics**: At 2.25 USD/bbl backwardation, storage can be profitable
- **Peak**: 5.75 USD/bbl (Jun 19) = moderate tightness
- **Implication**: Inventory levels moderate, not critically low

**Conclusion:** M1-M6 suggests moderate inventory levels. Storage sometimes profitable (unlike LSGO).

---

**Analysis:**

High M1-M6 means the market strongly values prompt supply. Storage becomes unattractive, inventories are likely low, and this reflects a structurally tight market, not just short-term noise.

---

### Chart 5: Z-Score of M1-M3

Quantifies short-term structure anomalies using rolling 60-day Z-scores.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

ax.plot(df_rolling['Date'], df_rolling['Z_M1_M3'], color='#9467bd', linewidth=1.5, label='Z-Score M1-M3')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
ax.axhline(y=2, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='Z=+2')
ax.axhline(y=3, color='gray', linestyle=':', linewidth=0.8, alpha=0.5, label='Z=+3')
ax.axhline(y=4, color='gray', linestyle='-.', linewidth=0.8, alpha=0.5, label='Z=+4')
ax.axhline(y=-2, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Date')
ax.set_ylabel('Z-Score')
ax.set_title('Z-Score of M1-M3 Spread (60-Day Rolling)', loc='left', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.show()

### Analysis: Statistical Anomaly Detection (M1-M3)

**Key Observations:**

- **Max Z-score**: 3.96 (Apr 30) = 4-sigma event
- **Frequency**: 1.7% of days had |Z| > 3 (vs 0.3% expected, vs 4.9% for LSGO)
- **Less extreme than LSGO**: 4-sigma vs 5-sigma, 1.7% vs 4.9% frequency

**Conclusion:** Brent shows moderate anomalies. Less extreme than LSGO diesel market.

---

**Analysis:**

- **Z > 2**: Notable tightening
- **Z > 3**: Strong anomaly
- **Z > 4**: Extreme structural stress

This turns visual intuition into a measurable framework for detecting abnormal regimes.

---

### Chart 6: Z-Score of M1-M6

Quantifies long-structure anomalies.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

ax.plot(df_rolling['Date'], df_rolling['Z_M1_M6'], color='#9467bd', linewidth=1.5, label='Z-Score M1-M6')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
ax.axhline(y=2, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='Z=+2')
ax.axhline(y=3, color='gray', linestyle=':', linewidth=0.8, alpha=0.5, label='Z=+3')
ax.axhline(y=4, color='gray', linestyle='-.', linewidth=0.8, alpha=0.5, label='Z=+4')
ax.axhline(y=-2, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Date')
ax.set_ylabel('Z-Score')
ax.set_title('Z-Score of M1-M6 Spread (60-Day Rolling)', loc='left', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.show()

### Analysis: Long-Structure Anomaly Detection (M1-M6)

**Key Observations:**

- **Max Z-score**: 3.50 (Jun 19) - same date as M1-M3 peak
- **Coherent stress**: Both short and long structure affected on same day
- **Interpretation**: Moderate structural tightness, not crisis

**Conclusion:** June 19 = moderate stress event affecting entire curve (but less severe than LSGO).

---

**Analysis:**

This reflects broader supply imbalance and storage tightness, not only prompt stress.

---

### Chart 7: Distribution of M1-M3 (Histogram)

Understanding the statistical distribution of the spread.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df_rolling['M1_M3'].dropna(), bins=50, color='#ff7f0e', alpha=0.7, edgecolor='black')
ax.axvline(x=df_rolling['M1_M3'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_rolling["M1_M3"].mean():.2f}')
ax.axvline(x=df_rolling['M1_M3'].median(), color='blue', linestyle='--', linewidth=2, label=f'Median: {df_rolling["M1_M3"].median():.2f}')

ax.set_xlabel('M1-M3 Spread (USD/bbl)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of M1-M3 Spread', loc='left', fontsize=12, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### Analysis: Spread Distribution & Fat Tails

**Key Observations:**

- **Skewness**: 1.02 (moderate right tail - backwardation more common)
- **Kurtosis**: 1.37 (**thin tails** - extreme events LESS frequent than normal)
- **Opposite of LSGO**: LSGO had fat tails (kurtosis 5.72), Brent has thin tails

**Conclusion:** Brent distribution is more normal than LSGO. Highly liquid global market.

---

**Analysis:**

The distribution shows right skew and fat tails. Extreme spreads occur more often than under a normal distribution, indicating that structural stress events are not rare.

---

### Chart 8: Spread vs Volatility Relationship

Scatter plot showing the relationship between structural tightness and volatility.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Remove NaN values
plot_data = df_rolling[['M1_M3', 'Vol_20D']].dropna()

ax.scatter(plot_data['M1_M3'], plot_data['Vol_20D'], alpha=0.5, s=20, color='#1f77b4')

ax.set_xlabel('M1-M3 Spread (USD/bbl)')
ax.set_ylabel('20D Annualized Volatility')
ax.set_title('Spread vs Volatility Relationship', loc='left', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Analysis: Structure-Volatility Relationship

**Key Observations:**

- **Correlation**: M1-M3 vs Vol = 0.516, M1-M6 vs Vol = 0.241
- **M1-M3 stronger**: Short-term structure drives volatility more than long structure
- **Opposite of LSGO**: LSGO had M1-M6 > M1-M3 (0.525 vs 0.476)

**Conclusion:** Brent volatility driven by prompt tightness (M1-M3), not structural imbalance (M1-M6).

---

**Analysis:**

Larger prompt spreads tend to coincide with higher volatility. This suggests that structure drives instability, and volatility is often endogenous to physical stress.

---

### Chart 9: Event Timeline

Overlay of M1-M3 spread and volatility with key market events.

In [ ]:
# Define key events (adjust based on actual market events)
events = {
    '2025-06-19': 'Peak backwardation',
    '2025-10-23': 'Elevated volatility'
}

fig, ax1 = plt.subplots(figsize=(13, 7))

# M1-M3 on left axis
ax1.plot(df_rolling['Date'], df_rolling['M1_M3'], color='#ff7f0e', linewidth=1.5, label='M1-M3')
ax1.set_xlabel('Date')
ax1.set_ylabel('M1-M3 Spread (USD/bbl)', color='#ff7f0e')
ax1.tick_params(axis='y', labelcolor='#ff7f0e')

# Volatility on right axis
ax2 = ax1.twinx()
ax2.plot(df_rolling['Date'], df_rolling['Vol_20D'], color='#d62728', linewidth=1.5, label='Volatility', alpha=0.7)
ax2.set_ylabel('20D Volatility', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')

# Add event markers
for event_date, event_label in events.items():
    ax1.axvline(x=pd.to_datetime(event_date), color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax1.text(pd.to_datetime(event_date), ax1.get_ylim()[1]*0.95, event_label, rotation=90, verticalalignment='top', fontsize=9)

ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.title('Event Timeline: Spread and Volatility', loc='left', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

### Analysis: Event-Driven Market Dynamics

**Key Events:**

- **Jan 15**: M1 peaked at 82.03 USD/bbl (highest point)
- **Apr 30**: Z-score hit 3.96 (4-sigma event)
- **Jun 19**: M1-M3 peaked at 3.27 USD/bbl, M1-M6 at 5.75 USD/bbl
- **Jul 08**: Volatility spiked to 52%
- **Dec 16**: M1 dropped to 58.92 USD/bbl (lowest point)

**Pattern**: Downward trend (Jan high to Dec low), moderate stress in April-June

**Conclusion**: Different from LSGO V-shape. Brent showed progressive decline with moderate mid-year stress.

---

**Analysis:**

Spread expansion should be analyzed together with external events. This helps distinguish genuine physical shocks from ordinary noise and connects price structure with real market drivers.

---

### Chart 10: Market Dashboard

Comprehensive view of all key metrics in a single dashboard.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Top-left: M1 Price
axes[0,0].plot(df_rolling['Date'], df_rolling['M1'], color='#1f77b4', linewidth=1.5)
axes[0,0].set_title('M1 Price (USD/bbl)', fontsize=11, fontweight='bold')
axes[0,0].set_ylabel('Price')
axes[0,0].grid(True, alpha=0.3)
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0,0].xaxis.set_major_locator(mdates.MonthLocator(interval=2))

# Top-right: M1-M3 Spread
axes[0,1].plot(df_rolling['Date'], df_rolling['M1_M3'], color='#ff7f0e', linewidth=1.5)
axes[0,1].axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
axes[0,1].axhline(y=30, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='Stress threshold')
axes[0,1].set_title('M1-M3 Spread (USD/bbl)', fontsize=11, fontweight='bold')
axes[0,1].set_ylabel('Spread')
axes[0,1].grid(True, alpha=0.3)
axes[0,1].legend(loc='upper left', fontsize=9)
axes[0,1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0,1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))

# Bottom-left: Z-Score M1-M3
axes[1,0].plot(df_rolling['Date'], df_rolling['Z_M1_M3'], color='#9467bd', linewidth=1.5)
axes[1,0].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[1,0].axhline(y=2, color='orange', linestyle='--', linewidth=0.8, alpha=0.5, label='Z=+2')
axes[1,0].axhline(y=3, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='Z=+3')
axes[1,0].axhline(y=4, color='darkred', linestyle='--', linewidth=0.8, alpha=0.5, label='Z=+4')
axes[1,0].set_title('Z-Score M1-M3 (60-Day Rolling)', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('Z-Score')
axes[1,0].grid(True, alpha=0.3)
axes[1,0].legend(loc='upper left', fontsize=9)
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1,0].xaxis.set_major_locator(mdates.MonthLocator(interval=2))

# Bottom-right: Volatility
axes[1,1].plot(df_rolling['Date'], df_rolling['Vol_20D'], color='#d62728', linewidth=1.5)
axes[1,1].axhline(y=df_rolling['Vol_20D'].mean(), color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label=f'Mean: {df_rolling["Vol_20D"].mean():.2f}')
axes[1,1].set_title('20-Day Annualized Volatility', fontsize=11, fontweight='bold')
axes[1,1].set_ylabel('Volatility')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].legend(loc='upper left', fontsize=9)
axes[1,1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1,1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))

plt.suptitle('LSGO Market Dashboard', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### Analysis: Dashboard Overview

**Key Insights:**

- **Price-spread decoupling**: Price declined while spreads widened (Apr-Jun)
- **Z-score early warning**: Crossed 2 in April (before June spread peak)
- **Volatility lag**: Peaked in July (after spread peak)
- **Moderate stress**: All metrics stayed in normal ranges

**Conclusion:** Dashboard shows moderate, manageable market stress. No crisis-level events.

---

### Chart 11: Regime Classification & Timeline

Systematic classification of market regimes based on M1-M6 spread.

In [ ]:
# Define regime classification function
def classify_regime(m1_m6):
    if pd.isna(m1_m6):
        return 'Unknown'
    elif m1_m6 > 100:
        return 'Crisis'
    elif m1_m6 > 50:
        return 'Stress'
    elif m1_m6 > 20:
        return 'Tight'
    else:
        return 'Normal'

# Apply classification
df_rolling['Regime'] = df_rolling['M1_M6'].apply(classify_regime)

# Count days per regime
regime_counts = df_rolling['Regime'].value_counts()
print('Days per regime:')
print(regime_counts)
print(f'\nPercentage breakdown:')
for regime in ['Normal', 'Tight', 'Stress', 'Crisis', 'Unknown']:
    if regime in regime_counts.index:
        pct = 100 * regime_counts[regime] / len(df_rolling)
        print(f'{regime:10s}: {regime_counts[regime]:3d} days ({pct:5.1f}%)')

In [ ]:
# Visualize regime timeline
fig, ax = plt.subplots(figsize=(14, 6))

# Define colors for each regime
regime_colors = {
    'Normal': '#2ca02c',    # Green
    'Tight': '#ff7f0e',     # Orange
    'Stress': '#d62728',    # Red
    'Crisis': '#8b0000',    # Dark red
    'Unknown': 'gray'      # Gray
}

# Plot M1-M6 with color-coded regimes
for regime in ['Normal', 'Tight', 'Stress', 'Crisis', 'Unknown']:
    mask = df_rolling['Regime'] == regime
    if mask.any():
        ax.scatter(df_rolling[mask]['Date'], df_rolling[mask]['M1_M6'], 
                  c=regime_colors[regime], label=regime, alpha=0.7, s=30)

# Add regime threshold lines
ax.axhline(y=20, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='Tight threshold')
ax.axhline(y=50, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Stress threshold')
ax.axhline(y=100, color='darkred', linestyle='--', linewidth=1, alpha=0.5, label='Crisis threshold')

ax.set_xlabel('Date')
ax.set_ylabel('M1-M6 Spread (USD/bbl)')
ax.set_title('Regime Classification Timeline', loc='left', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', ncol=2)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.show()

### Analysis: Regime Classification

**Key Observations:**

- **Normal regime only**: 67% of days (193/288), rest Unknown (NaN in M1-M6)
- **Never reached Tight/Stress/Crisis**: M1-M6 max = 5.75 USD/bbl (threshold = 20)
- **Stark contrast with LSGO**: LSGO had Crisis regime (2.1% of days)

**Conclusion:** Brent remained in Normal regime throughout. Much more stable than LSGO.

---

### Note: Comparative Context

**Question:** Is 99.7% backwardation normal for crude oil markets?

**Context (General Market Knowledge):**

Crude Oil and crude oil markets typically operate in backwardation due to:
- Just-in-time logistics (low storage capacity)
- Seasonal demand patterns (winter heating, summer agriculture)
- High storage costs relative to product value
- Refinery economics (crude oil is a middle distillate, not easily stored)

**Typical Backwardation Levels:**
- **Normal years**: 60-80% of days in backwardation
- **Tight years**: 85-95% of days in backwardation
- **2025 Brent Crude**: 99.7% of days in backwardation ← **Unusually high**

**Interpretation:**
- 2025 appears to be an **atypically tight year** for European crude oil
- Persistent backwardation suggests structural supply issues
- Possible drivers: refinery capacity constraints, geopolitical factors, demand strength

**Caveat:**
- Without multi-year Brent Crude data, we cannot definitively confirm this is anomalous
- Comparison to other markets (US ULSD, Singapore crude oil) would provide better context
- This analysis is limited to the 13-month sample period

**Future Work:**
- Extend analysis to 2020-2024 data to establish historical baseline
- Compare Brent Crude to other crude oil benchmarks
- Integrate fundamental data (EIA inventories, refinery runs) to confirm drivers

---

### 5.1 Market Structure: Moderate Backwardation & Storage Economics

**Backwardation Dominance:**
- Market in backwardation **88.9% of the time** (256 out of 288 days)
- Average M1-M3 spread: **1.14 USD/bbl**
- Average M1-M6 spread: **2.25 USD/bbl**
- Contango observed: 32 days (M1-M3 flat or negative)

**Storage Economics:**
- At 2.25 USD/bbl average backwardation, storage is moderately attractive
- Typical storage cost: 0.5-1.0 USD/bbl for 6 months
- Net profit from storage: ~1.50 USD/bbl (positive)
- At peak (5.75 USD/bbl), storage becomes very uneconomic

**Interpretation:**
- This is **more normal** than LSGO - indicates moderate supply management
- Global crude oil market operates with more flexibility than regional diesel
- Storage plays are sometimes profitable (unlike LSGO where always unprofitable)
- Inventory levels were moderate throughout the period
- Market has buffer capacity to absorb shocks

**Market Implication:**
- Storage plays (buying prompt, selling deferred) can be profitable
- Stocks are moderate, not structurally low
- The market operates with safety margin

---

### Summary Table: Key Metrics

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **Backwardation frequency** | 88.9% (256/288 days) | Moderate supply tightness |
| **M1-M3 average** | 1.14 USD/bbl | Mild backwardation |
| **M1-M6 average** | 2.25 USD/bbl | Storage moderately attractive |
| **Max M1-M3** | 3.27 USD/bbl (Jun 19, 2025) | 2.9x average |
| **Max M1-M6** | 5.75 USD/bbl (Jun 19, 2025) | 2.6x average |
| **Max Z-score** | 3.96 (Apr 30, 2025) | 4-sigma event |
| **Max volatility** | 52% annualized (Jul 08, 2025) | 1.8x average |
| **Skewness** | 1.02 | Moderate right tail |
| **Kurtosis** | 1.37 | Thin tails (normal-like) |
| **Correlation (M1-M3 vs Vol)** | 0.516 | Structure drives volatility |
| **Correlation (M1-M6 vs Vol)** | 0.241 | Weaker for long structure |
| **Days with Z > 3** | 5 (1.7%) | vs 0.3% expected |
| **Days with Z > 4** | 0 (0.0%) | vs 0.006% expected |

---



### Summary Table: Key Metrics

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **Backwardation frequency** | 99.7% (287/288 days) | Chronic supply tightness |
| **M1-M3 average** | 18.4 USD/MT | Persistent prompt scarcity |
| **M1-M6 average** | 33.7 USD/MT | Storage uneconomic |
| **Max M1-M3** | 85.25 USD/MT (June 19) | 4.6x average |
| **Max M1-M6** | 118.5 USD/MT (July 9) | 3.5x average |
| **Max Z-score** | 5.25 (June 19) | 5-sigma event |
| **Max volatility** | 78% annualized (July 10) | 2.2x average |
| **Skewness** | 2.12 | Strong right tail |
| **Kurtosis** | 5.72 | Fat tails (16x more extremes) |
| **Correlation (M1-M3 vs Vol)** | 0.476 | Structure drives volatility |
| **Correlation (M1-M6 vs Vol)** | 0.525 | Stronger for long structure |
| **Days with Z > 3** | 14 (4.9%) | vs 0.3% expected |
| **Days with Z > 4** | 6 (2.1%) | vs 0.006% expected |

---



---

## At a Glance: The June 2025 Crisis

### Timeline
```
April 9, 2025    →  M1 drops to 579 USD/MT (lowest point)
June 10, 2025    →  Early warning: Z-score = 2.8, M1-M3 = 45 USD/MT
June 19, 2025    →  PEAK: Z-score = 5.25, M1-M3 = 85.25 USD/MT
July 9, 2025     →  M1-M6 peaks at 118.5 USD/MT
July 10, 2025    →  Volatility spikes to 78% annualized
Oct 2025         →  Gradual normalization begins
Jan 2026         →  Return to mild backwardation
```

### By the Numbers
- **38% price swing** (579 → 798.5 USD/MT)
- **5-sigma event** (should occur once every 5,000 years)
- **4.6x average** M1-M3 spread
- **9 days advance warning** from Z-score framework
- **Weeks of persistence** (not a single-day spike)

### What It Means
- **Genuine supply crisis** (not speculative noise)
- **Entire curve affected** (short + long structure)
- **Fat tails confirmed** (extreme events 16x more frequent)
- **Framework worked** (early warning signals detected)
- **Slow mean reversion** (took months to normalize)

---



---

## At a Glance: Brent 2025 Market Dynamics

### Timeline
```
Jan 15, 2025    →  M1 peaks at 82.03 USD/bbl (highest point)
Apr 30, 2025    →  PEAK Z-score = 3.96 (4-sigma event)
Jun 19, 2025    →  Max M1-M3 = 3.27 USD/bbl
Jul 08, 2025    →  Volatility spikes to 52% annualized
Dec 16, 2025    →  M1 drops to 58.92 USD/bbl (lowest point)
```

### By the Numbers
- **39.2% price swing** (58.92 to 82.03 USD/bbl)
- **4-sigma event** (Z-score 3.96 on Apr 30, 2025)
- **2.9x average** M1-M3 spread at peak
- **5 days** with |Z| > 3 (1.7% vs 0.3% expected)
- **Downward trend** (Jan high to Dec low)

### What It Means
- **Moderate market stress** (less extreme than LSGO diesel)
- **Normal distribution** (thin tails, not fat tails)
- **Liquid global market** (Brent more flexible than regional diesel)
- **Different pattern** (downward trend vs LSGO V-shape)
- **Structure-volatility link confirmed** (correlation 0.52)

---



## 6. Conclusions

### 6.1 Summary of Findings

This analysis of the Brent Crude forward curve from January 2025 to February 2026 reveals:

1. **Chronic supply tightness** (99.7% backwardation)
2. **Extreme stress episode** (June-July 2025: 5-sigma event)
3. **Fat-tailed distribution** (extreme events 16x more frequent than normal)
4. **Structure-driven volatility** (0.48-0.53 correlation)
5. **Regime clustering** (stress events are not random)
6. **Unattractive storage economics** (persistent backwardation)

The European crude oil market operates with minimal inventory buffers and high vulnerability to supply disruptions.

---

### 6.2 Trading & Risk Management Framework

**Systematic Monitoring:**
- Daily tracking of M1-M3, M1-M6 spreads
- 60-day rolling Z-scores
- 20-day rolling volatility
- Regime classification (Normal/Tight/Stress/Crisis)

**Trading Signals:**
- Z > 2: Monitor closely
- Z > 3: Reduce exposure or hedge
- Z > 4: Extreme stress, fundamentals dominate
- Spread widening = early warning for volatility increase

**Risk Management:**
- Adjust position sizing based on regime
- Implement tail hedges when Z > 3
- Account for fat tails in VaR calculations
- Expect slow mean reversion after extremes

---

### 6.3 Example Trade Setup: June 2025 Crisis

**Date:** June 10, 2025

**Market Observation:**
- M1-M3 = 45 USD/MT (rising from 20 USD/MT in May)
- Z-score M1-M3 = 2.8 (approaching 3)
- Volatility = 35% (rising from 25%)
- Regime: Transitioning from Tight to Stress

**Signal Interpretation:**
- Strong tightening in progress
- Z approaching extreme territory
- Early warning of potential crisis

**Recommended Actions:**
1. **Reduce long flat price exposure** (risk of volatility spike)
2. **Consider selling M1-M2 calendar spread** (capture backwardation)
3. **Buy volatility** (expect further increase)
4. **Set alert for Z > 4** (crisis regime, fundamentals dominate)

**Actual Outcome:**
- June 19: Z hit 5.25, M1-M3 reached 85.25 USD/MT
- July 10: Volatility spiked to 78%
- Early signal (June 10) provided **9 days advance warning**

**Lesson:**
- Z-score framework provided actionable early warning
- Spread monitoring outperformed volatility as leading indicator
- Framework would have triggered risk reduction before crisis peak

---

### 6.4 Limitations & Caveats

**Data Limitations:**
- **13-month sample is short** for robust statistical inference
- No fundamental data (inventories, refinery runs, demand) to confirm physical drivers
- Event labels (Chart 9) are inferred from price data, not verified against news/events
- Cannot determine if 2025 is typical or anomalous without multi-year comparison

**Methodological Limitations:**
- **Z-score window (60 days) is arbitrary** - not optimized or backtested
- Normal distribution assumption for Z-scores (ironic given we prove fat tails)
- No out-of-sample testing for regime classification
- Correlation does not prove causation (though structure → volatility is economically sound)

**Market Context:**
- 2025 may be atypical (geopolitical events, refinery issues, etc.)
- European crude oil has unique characteristics (not generalizable to other markets)
- Storage costs and logistics vary by region

**What This Means:**
- **Findings are directionally correct** but magnitudes may vary with more data
- **Framework is sound** but parameters need calibration and validation
- **This is exploratory analysis**, not a production trading system
- Further work needed: multi-year data, fundamental integration, backtesting

**Honest Assessment:**
- We've identified patterns, but cannot claim predictive power without out-of-sample testing
- The June crisis analysis is retrospective - we don't know if signals would work in real-time
- More data and rigorous validation needed before deploying capital

---

### 6.5 Final Thoughts

This analysis demonstrates that **forward curve structure contains rich information** about physical market fundamentals. The spreads between contract months encode:

- Supply-demand balance
- Inventory levels
- Storage economics
- Market stress

By systematically analyzing these spreads, we can:

1. **Detect structural stress early** (before it appears in volatility)
2. **Quantify abnormal regimes** (using Z-scores and distribution analysis)
3. **Build actionable frameworks** (regime-based monitoring)
4. **Improve risk management** (accounting for fat tails and clustering)

The June-July 2025 crisis illustrates why this matters: a 5-sigma event that "should" occur once every 5,000 years happened in a 13-month sample. **Markets are not normal. Tail risk is real.** And the forward curve provides early warning signals.

**For commodity traders, risk managers, and systematic strategists, forward curve analysis is essential.**

---

## 7. Technical Note

This analysis applies the same methodology as the LSGO (diesel) project: rolling contract construction with proper expiry handling, statistical anomaly detection (Z-scores, distribution analysis), regime classification frameworks, and forward curve as leading indicator. The methodology combines physical commodity intuition with quantitative rigor, producing actionable signals for trading and risk management. Code and framework are production-ready and applicable to other commodity markets. All analysis is based on actual market data (Jan 2025 - Feb 2026) with transparent limitations and caveats.

**Comparison with LSGO:** Results will be compared with diesel market findings in Project 3 to identify correlations, divergences, and market-specific dynamics.

---